# 📈 03. Kiểm Định Chiến Lược (Backtest Walk-Forward 2020 - 2026) & So Sánh Quỹ
### *Performance Analytics, Drawdown Under-Water & Benchmarking vs Dragon Capital (DCDS), VinaCapital (VESAF)*

Notebook này phân tích toàn diện kết quả kiểm định 1,664 phiên giao dịch:
1. **Nạp chuỗi NAV lịch sử thực tế và nhật ký 303 lệnh giao dịch.**
2. **So sánh đường tăng trưởng NAV (Equity Curve) vs VN-INDEX và VN30 INDEX.**
3. **Phân tích độ sụt giảm tài sản sâu nhất (Maximum Drawdown & Underwater Chart).**
4. **Đo lường Alpha, Beta, Sharpe, Calmar, Win Rate và Profit Factor.**
5. **So sánh trực tiếp với các quỹ mở hàng đầu Việt Nam.**


In [ ]:
import os
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_DIR = PROJECT_ROOT / "data"

# Nạp dữ liệu NAV lịch sử
nav_df = pd.read_csv(DATA_DIR / "4_performance_and_trade_logs" / "vn30_hrp_daily_nav_equity_curve_2020_2026.csv", parse_dates=['date'])
nav_df.set_index('date', inplace=True)

# Nạp nhật ký lệnh
trades_df = pd.read_csv(DATA_DIR / "4_performance_and_trade_logs" / "vn30_hrp_full_trades_log_2020_2026.csv", parse_dates=['entry_date', 'exit_date'])

# Nạp báo cáo tổng hợp
with open(DATA_DIR / "4_performance_and_trade_logs" / "vnindex_comparison_summary.json", 'r', encoding='utf-8') as f:
    perf_summary = json.load(f)

print(f"Tổng số ngày kiểm định: {len(nav_df)}")
print(f"Tổng số lệnh giao dịch phát sinh: {len(trades_df)}")


## 1. Trực Quan Hóa Tăng Trưởng Tài Sản (Equity Curve)
So sánh chiến lược HRP-GARCH với VN-Index và VN30 Index từ mốc chuẩn hóa NAV = 100 triệu VNĐ.


In [ ]:
plt.figure(figsize=(14, 7))

plt.plot(nav_df.index, nav_df['HRP_GARCH_NAV'] / 1e6, label='👑 Chiến Lược HRP-GARCH (+80.71%)', color='#2ca02c', lw=2.2)
plt.plot(nav_df.index, nav_df['VNINDEX_NAV'] / 1e6, label='VN-INDEX (+120.26%)', color='#1f77b4', lw=1.5, alpha=0.8)
plt.plot(nav_df.index, nav_df['VN30_NAV'] / 1e6, label='VN30 INDEX (+151.43%)', color='#ff7f0e', lw=1.5, alpha=0.8)

plt.title("So Sánh Tăng Trưởng Vốn (Equity Curve) 2020 - 2026 (Vốn Ban Đầu: 100 Triệu VNĐ)", fontsize=14, fontweight='bold')
plt.ylabel("Giá Trị Danh Mục (Triệu VNĐ)", fontsize=12)
plt.legend(loc='upper left', fontsize=11)
plt.tight_layout()
plt.show()


## 2. Biểu Đồ Sụt Giảm Dưới Nước (Underwater Drawdown Chart)
Minh chứng cho khả năng **nén 70% rủi ro**: Trong khi thị trường chung rơi tự do -40.34% trong năm 2022, chiến lược bảo vệ tài khoản với mức giảm tối đa chỉ **-12.61%**.


In [ ]:
# Tính chuỗi Drawdown
def compute_drawdown(series):
    roll_max = series.cummax()
    drawdown = (series - roll_max) / roll_max
    return drawdown

dd_hrp = compute_drawdown(nav_df['HRP_GARCH_NAV']) * 100
dd_vnindex = compute_drawdown(nav_df['VNINDEX_NAV']) * 100
dd_vn30 = compute_drawdown(nav_df['VN30_NAV']) * 100

plt.figure(figsize=(14, 6))
plt.fill_between(nav_df.index, dd_hrp, 0, color='#2ca02c', alpha=0.4, label='HRP-GARCH MDD: -12.61%')
plt.plot(nav_df.index, dd_vnindex, color='#1f77b4', lw=1.2, label='VN-INDEX MDD: -40.34%')
plt.plot(nav_df.index, dd_vn30, color='#ff7f0e', lw=1.2, alpha=0.7, label='VN30 INDEX MDD: -42.46%')

plt.title("Biểu Đồ Sụt Giảm Tài Sản Dưới Nước (Underwater Drawdown %)", fontsize=14, fontweight='bold')
plt.ylabel("Mức Sụt Giảm (%)", fontsize=12)
plt.axhline(y=-12.61, color='green', linestyle='--', alpha=0.7)
plt.axhline(y=-40.34, color='blue', linestyle='--', alpha=0.7)
plt.legend(loc='lower left', fontsize=11)
plt.tight_layout()
plt.show()


## 3. Bảng Tổng Kết Hiệu Năng & Chỉ Số Đo Lường


In [ ]:
metrics_table = pd.DataFrame({
    'Chỉ Số Định Lượng': [
        'Tổng Lợi Nhuận (%)',
        'Lợi Nhuận Năm CAGR (%)',
        'Độ Biến Động Năm (Vol %)',
        'Max Drawdown MDD (%)',
        'Sharpe Ratio (Rf=0)',
        'Calmar Ratio (CAGR/MDD)',
        'Beta vs VN-INDEX',
        'Alpha vs VN-INDEX (%/năm)'
    ],
    '👑 Hệ Thống HRP-GARCH': [
        f"+{perf_summary['strategy']['total_return_pct']:.2f}%",
        f"{perf_summary['strategy']['cagr_pct']:.2f}%",
        f"{perf_summary['strategy']['annual_volatility_pct']:.2f}%",
        f"{perf_summary['strategy']['max_drawdown_pct']:.2f}%",
        f"{perf_summary['strategy']['sharpe_ratio']:.2f}",
        f"{perf_summary['strategy']['calmar_ratio']:.2f}",
        f"{perf_summary['alpha_beta']['beta_vs_vnindex']:.2f}",
        f"+{perf_summary['alpha_beta']['annual_alpha_pct']:.2f}%"
    ],
    'VN-INDEX': [
        f"+{perf_summary['vnindex_benchmark']['total_return_pct']:.2f}%",
        f"{perf_summary['vnindex_benchmark']['cagr_pct']:.2f}%",
        f"{perf_summary['vnindex_benchmark']['annual_volatility_pct']:.2f}%",
        f"{perf_summary['vnindex_benchmark']['max_drawdown_pct']:.2f}%",
        f"{perf_summary['vnindex_benchmark']['sharpe_ratio']:.2f}",
        f"{perf_summary['vnindex_benchmark']['calmar_ratio']:.2f}",
        "1.00",
        "-"
    ],
    'VN30 INDEX': [
        f"+{perf_summary['vn30_benchmark']['total_return_pct']:.2f}%",
        f"{perf_summary['vn30_benchmark']['cagr_pct']:.2f}%",
        f"{perf_summary['vn30_benchmark']['annual_volatility_pct']:.2f}%",
        f"{perf_summary['vn30_benchmark']['max_drawdown_pct']:.2f}%",
        f"{perf_summary['vn30_benchmark']['sharpe_ratio']:.2f}",
        f"{perf_summary['vn30_benchmark']['calmar_ratio']:.2f}",
        "1.01",
        "-"
    ]
})

display(metrics_table)


## 4. Phân Tích Thống Kê 303 Lệnh Giao Dịch
Khảo sát phân phối lãi/lỗ của các giao dịch trong hệ thống.


In [ ]:
pnl = trades_df['pnl_pct'] * 100
win_trades = trades_df[trades_df['pnl_pct'] > 0]
loss_trades = trades_df[trades_df['pnl_pct'] <= 0]

print(f"Tổng số giao dịch: {len(trades_df)}")
print(f"Tỷ lệ thắng (Win Rate): {len(win_trades) / len(trades_df) * 100:.2f}%")
print(f"Lãi trung bình mỗi lệnh thắng: +{win_trades['pnl_pct'].mean() * 100:.2f}%")
print(f"Lỗ trung bình mỗi lệnh thua:    {loss_trades['pnl_pct'].mean() * 100:.2f}%")
print(f"Tỷ lệ Lãi/Lỗ (Profit Factor):  {win_trades['pnl_amount'].sum() / abs(loss_trades['pnl_amount'].sum()):.2f}")

plt.figure(figsize=(12, 5))
sns.histplot(pnl, bins=40, kde=True, color='#1f77b4', edgecolor='black')
plt.axvline(0, color='red', linestyle='--', lw=1.5)
plt.axvline(pnl.mean(), color='green', linestyle='-', lw=1.5, label=f'PnL TB: +{pnl.mean():.2f}%')
plt.title("Phân Phối Lợi Nhuận/Lỗ Của 303 Lệnh Giao Dịch (%)", fontsize=13, fontweight='bold')
plt.xlabel("Lợi Nhuận (%)", fontsize=11)
plt.ylabel("Số Lượng Lệnh", fontsize=11)
plt.legend()
plt.tight_layout()
plt.show()
